# Clase 2: Fundamentos de la Imagen Digital

En este encuentro vamos a cerrar nuestra etapa con py5 explorando algo central para todo el curso: **¿qué es realmente una imagen para una computadora?**

La respuesta corta es que una imagen no es más que una matriz de números. Cada píxel ocupa una posición `(x, y)` en el plano y guarda tres valores: la intensidad de los canales de color rojo (R), verde (G) y azul (B), cada uno entre 0 y 255.

Lo que vamos a hacer en la clase es comprobarlo de manera visual e interactiva, manipulando esos números directamente.

> **Sobre esta guía:** Los bloques de código de esta libreta están listos para copiar y pegar en un script de Python. La ejecución se hace desde una terminal, no desde el IDE. El proceso está detallado en la sección siguiente.

## 1. Preparación: entorno virtual y ejecución desde terminal

Para trabajar de forma ordenada, vamos a crear un entorno virtual. Esto asegura que las librerías que instalemos no interfieran con otros proyectos en la misma computadora.

### Crear e instalar el entorno

Abrí una terminal en la carpeta del proyecto y ejecutá estos comandos **una sola vez**:

```bash
python -m venv venv
pip install py5
```

### Activar el entorno (cada vez que trabajen)

**Windows:**
```bash
venv\Scripts\activate
```
**Linux / macOS:**
```bash
source venv/bin/activate
```
Cuando el entorno esté activo, van a ver el prefijo `(venv)` al inicio del prompt de la terminal.

### Crear y ejecutar los scripts

El flujo de trabajo para cada actividad es sencillo:

1. Abrir el IDE (VS Code o el que usen).
2. Crear un archivo nuevo con el nombre indicado en cada actividad.
3. Copiar y pegar el código de esta guía.
4. **Cerrar el IDE.**
5. Desde la terminal con el entorno activado, ejecutar:
   ```bash
   python nombre_del_archivo.py
   ```

Cerrar el IDE antes de ejecutar evita conflictos con las ventanas gráficas que genera py5. Es una práctica recomendable mientras trabajemos con sketches interactivos.

## 2. Actividad 1: La lupa de píxeles

### ¿Qué vamos a hacer?

Vamos a construir un inspector visual: al mover el mouse sobre una imagen, el programa va a leer el color exacto del píxel que está bajo el cursor, mostrarlo ampliado en un cuadrado a la derecha y desplegar sus valores R, G y B en pantalla.

El concepto clave es que cada coordenada `(x, y)` de la imagen corresponde a un píxel que almacena tres números. Ese es el fundamento de toda operación de procesamiento digital de imágenes.

### Referencia a sketches anteriores

Este ejercicio combina lo que ya hicieron en:
- `005_upload_img.py`: carga de una imagen desde archivo.
- `006_pixeles.py`: acceso a la matriz de píxeles.
- `009_mouse.py`: uso de las coordenadas del mouse.

### Código

Creá un archivo llamado `lupa.py` y copiá este bloque completo:

```python
import py5

img = None

def setup():
    global img
    py5.size(800, 400)
    img = py5.load_image("img/imagen.jpg")  # Usá una imagen disponible en tu carpeta img/
    img.resize(400, 400)

def draw():
    py5.background(255)

    # Mostrar la imagen en la mitad izquierda
    py5.image(img, 0, 0)

    # Limitar las coordenadas del mouse al área de la imagen
    # Esto evita errores si el cursor sale de la imagen
    mx = py5.constrain(py5.mouse_x, 0, 399)
    my = py5.constrain(py5.mouse_y, 0, 399)

    # Obtener el color del píxel en esa posición
    color_pixel = py5.get_pixels(int(mx), int(my))

    # Separar el color en sus tres canales
    r = py5.red(color_pixel)
    g = py5.green(color_pixel)
    b = py5.blue(color_pixel)

    # Mostrar el color como un cuadrado en la mitad derecha (la "lupa")
    py5.fill(color_pixel)
    py5.stroke(0)
    py5.rect(450, 50, 300, 300)

    # Mostrar los valores numéricos
    py5.fill(0)
    py5.text_size(18)
    py5.text(f"Posición: ({mx}, {my})", 450, 30)
    py5.text(f"R: {r:.0f}   G: {g:.0f}   B: {b:.0f}", 450, 380)

py5.run_sketch()
```

### ¿Qué hace cada parte?

| Línea / función | Explicación |
|---|---|
| `img.resize(400, 400)` | Ajusta la imagen a un tamaño fijo para que las coordenadas sean predecibles. |
| `py5.constrain(valor, min, max)` | Limita un número dentro de un rango. Acá asegura que el mouse no acceda a posiciones fuera de la imagen, lo que causaría un error. |
| `py5.get_pixels(x, y)` | Lee el color del píxel en la posición `(x, y)`. Devuelve un valor compacto que contiene R, G y B juntos. |
| `py5.red()`, `py5.green()`, `py5.blue()` | Extraen cada canal de color por separado a partir del color compacto. |
| `py5.fill(color_pixel)` | Establece el color de relleno para las figuras que se dibujen a continuación. |

### Para experimentar

Una vez que el sketch funcione, probá estas variaciones. Modificá una cosa a la vez y observá el resultado:

1. **Color negativo:** Reemplazá `py5.fill(color_pixel)` por `py5.fill(255 - r, 255 - g, 255 - b)`. El color del cuadrado debería ser el complementario del píxel original. ¿Qué color aparece sobre un rojo puro? ¿Y sobre el blanco?

2. **Aislamiento de canal:** Ahora usá `py5.fill(r, 0, 0)`. Esto elimina verde y azul y muestra solo la contribución del canal rojo. Pasá el cursor por zonas azules o verdes de la imagen y observá cuánto rojo tienen en realidad.

3. **Sin protección:** Comentá la línea con `py5.constrain()` reemplazándola por `mx = py5.mouse_x` y `my = py5.mouse_y`. Mové el mouse fuera de la imagen rápidamente. ¿Qué mensaje de error aparece en la terminal? ¿Qué tipo de error es?

## 3. Actividad 2: Mezclador de canales con el mouse

### ¿Qué vamos a hacer?

Vamos a recorrer la matriz de píxeles de una imagen y modificar matemáticamente sus valores antes de mostrarla. La posición horizontal del mouse va a controlar la intensidad del canal rojo: a la izquierda, el canal rojo se suprime; a la derecha, se amplifica.

Este ejercicio introduce el concepto de **filtro**: una operación matemática que transforma los valores de los píxeles de manera sistemática. Es la base de operaciones como el contraste, la corrección de color y la separación de canales.

### Referencia a sketches anteriores

- `003_RGB.py`: separación visual de canales.
- `006_pixeles.py` y `007_pixeles.py`: recorrido de la matriz de píxeles.
- `008_filtro.py`: aplicación de filtros con `tint`.

### Código

Creá un archivo llamado `canal_mouse.py`:

```python
import py5

img = None

def setup():
    global img
    py5.size(800, 400)
    img = py5.load_image("img/imagen.jpg")
    img.resize(400, 400)

def draw():
    py5.background(35)

    # Imagen original en la mitad izquierda (sin modificar)
    py5.image(img, 0, 0)

    # Calcular el factor de ajuste según la posición X del mouse
    # remap convierte un valor de un rango a otro
    # Acá: de 0 a 800 píxeles de ancho → de 0 a 2.5 de factor multiplicador
    factor_rojo = py5.remap(py5.mouse_x, 0, py5.width, 0, 2.5)

    # Acceder a la matriz de píxeles del lienzo completo
    img.load_pixels()
    py5.load_pixels()

    for x in range(img.width):
        for y in range(img.height):

            # La imagen es un arreglo lineal. Para acceder al píxel (x, y):
            # índice = x + y * ancho
            indice_img = x + y * img.width
            pixel = img.pixels[indice_img]

            # Separar los canales
            r = py5.red(pixel)
            g = py5.green(pixel)
            b = py5.blue(pixel)

            # Modificar solo el canal rojo según el mouse
            r = r * factor_rojo

            # Limitar el valor para que no supere 255
            # Un valor mayor haría que py5 lo interprete incorrectamente
            if r > 255:
                r = 255

            # Calcular el índice del mismo píxel en el lienzo (desplazado 400px a la derecha)
            indice_canvas = (x + 400) + y * py5.width
            py5.pixels[indice_canvas] = py5.color(r, g, b)

    # Aplicar los cambios al lienzo
    py5.update_pixels()

py5.run_sketch()
```

### ¿Qué hace cada parte?

| Línea / función | Explicación |
|---|---|
| `py5.remap(v, a1, a2, b1, b2)` | Convierte un valor del rango `[a1, a2]` al rango `[b1, b2]`. Es análogo a una regla de tres. Acá transforma la posición X del mouse en un factor multiplicador entre 0 y 2.5. |
| `índice = x + y * ancho` | Fórmula fundamental: convierte coordenadas 2D en un índice de arreglo lineal. Una imagen de 400×400 tiene 160.000 píxeles almacenados en fila. |
| `img.load_pixels()` | Carga la imagen en memoria para poder leer sus valores. Es obligatorio antes de acceder a `img.pixels[]`. |
| `py5.update_pixels()` | Confirma y renderiza los cambios realizados en el arreglo `py5.pixels[]`. Sin esta línea, los cambios no se reflejan en pantalla. |
| `if r > 255: r = 255` | Recorte o clipping: evita que el valor desborde el rango de 8 bits. Sin esto, la asignación de color se comporta de forma inesperada. |

### Para experimentar

1. **Suprimir el canal rojo por completo:** Reemplazá `r = r * factor_rojo` por `r = 0`. La imagen de la derecha debería mostrar solo los canales verde y azul. ¿Qué pasa con las zonas que eran originalmente rojas?

2. **Intercambiar canales:** Cambiá `py5.color(r, g, b)` por `py5.color(b, g, r)`. Esto intercambia el canal rojo y el azul. Los cielos de color azul deberían volverse rojizos. Pensá qué implica esto: los colores son datos, y cambiar su posición genera una imagen que parece incorrecta pero matemáticamente es válida.

3. **Controlar un canal distinto:** En lugar de modificar `r`, aplicá el factor al canal verde (`g = g * factor`). Considerá también crear un segundo factor que use la posición Y del mouse para controlar el azul.